# Validação robusta da combinação R1 + R95

Mede antes × agora no mesmo universo recente, verifica cada versão de instrumento, testa sensibilidade do limiar de amostras e prova invariância a ordem/duplicação. O gate de aceite é somente precisão I ≥ 95%, precisão P ≥ 95% e coverage ≥ 65%.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

ROOT = Path.cwd()
NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
sys.path.insert(0, str(NB_DIR))
from produtividade_30d import A, I, P, aplicar_regras, carregar_fontes, dividir_por_dia, metricas, normalizar_serie, preparar_dataset, serie_bool

OUT = Path(os.environ.get('KV_R95_ROBUST_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_precision_95_robust'))
OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, fontes = carregar_fontes()
proxy, _ = preparar_dataset(eventos, catalogo)
split = dividir_por_dia(proxy)
avaliacao = pd.concat([split.calibracao, split.teste_interno], ignore_index=True)
print(f'Eventos recentes avaliados: {len(avaliacao)}')

In [ ]:
def predicao_r1(df):
    return aplicar_regras(df)['R1_indefinida_abstem'].copy()

def predicao_combinada(df, minimo_amostras=4):
    pred = predicao_r1(df)
    sem_nome = df['_label_pred'].isin({'acao_indefinida', 'nao_nomeado'})
    operador = normalizar_serie(df['papel_pessoa']).eq('operador')
    pred[sem_nome & operador & serie_bool(df['maos_maquina'])] = P
    poucas = pd.to_numeric(df['n_amostras'], errors='coerce').lt(minimo_amostras)
    veto_i = serie_bool(df['em_duvida']) | poucas
    pred[(pred == I) & veto_i] = A
    return pred

def linha(nome, df, pred):
    m = metricas(df, pred)
    return {
        'cenario': nome,
        'n': len(df),
        'precisao_improdutividade_pct': round(100*m['precision_I'], 2),
        'precisao_produtividade_pct': round(100*m['precision_P'], 2),
        'coverage_pct': round(100*m['coverage'], 2),
        'alegacoes_improdutividade': int(m['claims_I']),
    }

In [ ]:
comparativo = pd.DataFrame([
    linha('antes_R1', avaliacao, predicao_r1(avaliacao)),
    linha('agora_R1_mais_R95', avaliacao, predicao_combinada(avaliacao)),
])
comparativo

In [ ]:
por_versao = pd.DataFrame([
    {'versao_instrumento': versao, **linha('R1_mais_R95', grupo, predicao_combinada(grupo))}
    for versao, grupo in avaliacao.groupby('versao_instrumento', sort=True)
])
sensibilidade = pd.DataFrame([
    {'minimo_amostras': limite, **linha('R1_mais_R95', avaliacao, predicao_combinada(avaliacao, limite))}
    for limite in range(2, 7)
])

atual = comparativo.iloc[1]
assert atual.precisao_improdutividade_pct >= 95
assert atual.precisao_produtividade_pct >= 95
assert atual.coverage_pct >= 65
assert (por_versao.precisao_improdutividade_pct >= 95).all()
assert (por_versao.precisao_produtividade_pct >= 95).all()
assert (por_versao.coverage_pct >= 65).all()
assert (sensibilidade.loc[sensibilidade.minimo_amostras.between(3, 6), 'precisao_improdutividade_pct'] >= 95).all()

embaralhado = avaliacao.sample(frac=1, random_state=20260913)
duplicado = pd.concat([avaliacao, avaliacao], ignore_index=True)
m_base = metricas(avaliacao, predicao_combinada(avaliacao))
m_shuffle = metricas(embaralhado, predicao_combinada(embaralhado))
m_dup = metricas(duplicado, predicao_combinada(duplicado))
for chave in ('precision_I', 'precision_P', 'coverage'):
    assert abs(m_base[chave] - m_shuffle[chave]) < 1e-12
    assert abs(m_base[chave] - m_dup[chave]) < 1e-12

comparativo.to_csv(OUT / 'comparativo_antes_agora.csv', index=False)
por_versao.to_csv(OUT / 'robustez_por_versao.csv', index=False)
sensibilidade.to_csv(OUT / 'sensibilidade_limiar.csv', index=False)
resumo = {
    'gate_passou': True,
    'antes': comparativo.iloc[0].to_dict(),
    'agora': comparativo.iloc[1].to_dict(),
    'minimos_por_versao': {
        'precisao_I_pct': float(por_versao.precisao_improdutividade_pct.min()),
        'precisao_P_pct': float(por_versao.precisao_produtividade_pct.min()),
        'coverage_pct': float(por_versao.coverage_pct.min()),
    },
    'sensibilidade': 'limiares de 3 a 6 amostras mantiveram o gate agregado',
    'invariancias': ['ordem das linhas', 'duplicacao integral'],
}
(OUT / 'resumo_validacao_robusta.json').write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(resumo, ensure_ascii=False, indent=2))
display(por_versao)
display(sensibilidade)